# Satellite-Derived Chlorophyll Time Series for Lakes: Sentinel

This notebook extracts chlorophyll-a concentration estimates and Normalized Difference Chlorophyll Index (NDCI) values from MODIS (Terra and Aqua) specified lake locations.

## Overview

- Purpose: Generate time series of chlorophyll indices from satellite imagery
- Study Areas: Detroit Lake and Upper Klamath Lake
- Satellite Sensors: Sentinel-2 provides high spatial (10-20m) and temporal (5-day revisit) resolution multispectral imagery since 2015.
  - Sentinel is a constellation of twin satellites (2A and 2B) offering 13 spectral bands optimized for vegetation and water monitoring.
- Output: CSV files with date-stamped chlorophyll index (NDCI) values

In [1]:
"""
Initialize Google Earth Engine (GEE) connection and authentication.

This cell sets up the GEE Python API for accessing satellite imagery collections.
Authentication is required on first use or when credentials expire.
"""

import ee
import math
import pandas as pd

# Authenticate and initialize GEE with your registered project
ee.Authenticate()
ee.Initialize(project='ee-toddsteissberg')  # Replace with your GEE Cloud Project ID

In [ ]:
# Sentinel-2 NDCI Extraction (following Johansen et al., 2024)

# -----------------------------------------------------------------------------
# Define lake locations and output filenames
# -----------------------------------------------------------------------------
lakes = [
    dict(name='Detroit',
         lon=-122.184, lat=44.711,
         export_id='Detroit_S2_NDCI_500m'),
    dict(name='UpperKlamath',
         lon=-121.900, lat=42.400,
         export_id='Klamath_S2_NDCI_500m')
]

# Temporal range (Sentinel-2 available from July 2015)
start_date, end_date = '2015-07-01', '2025-12-31'

# Spatial buffer for 500 m x 500 m Region of Interest (ROI)
half_size_m = 250  # metres

# Sentinel-2 Level-2A Surface Reflectance (harmonized)
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

In [ ]:
# -----------------------------------------------------------------------------
# Preprocessing and Index Functions
# -----------------------------------------------------------------------------

def mask_s2_scl_water(img):
    """
    Keep only SCL water (class 6), following Johansen et al. (2024), eroded
    by 1 pixel to reduce shoreline adjacency, and remove edge artifacts
    where B8A == 0.
    """
    scl = img.select('SCL')
    water = scl.eq(6)
    # Erode 1 pixel (20 m) to avoid shoreline bleed
    water_eroded = water.focal_min(1)
    edge_ok = img.select('B8A').gt(0)
    mask = water_eroded.And(edge_ok)
    return img.updateMask(mask)

def add_ndci(img):
    """
    NDCI = (B5 - B4) / (B5 + B4)
      - B5: 705 nm (20 m)
      - B4: 665 nm (10 m) -> resample & reproject to B5 grid
    Cast to float to avoid integer division; NDCI is scale-invariant.
    """
    b5 = img.select('B5').toFloat()  # 20 m native
    # Resample B4 to match B5's 20 m grid and projection
    b4 = (img.select('B4')
            .toFloat()
            .resample('bilinear')
            .reproject(b5.projection()))
    ndci = b5.subtract(b4).divide(b5.add(b4)).rename('NDCI')
    return img.addBands(ndci)

def preprocess_s2(img):
    """
    Preprocessing follows Johansen et al., 2024:
      1) SCL water-only (class 6) with 1 px erosion
      2) Edge artifact removal via B8A > 0
    """
    return mask_s2_scl_water(img)

def img_to_feature(img, roi, tag):
    """
    Convert image to a Feature with mean NDCI over ROI (20 m).
    Returns a Feature even if NDCI is null. We will drop nulls later.
    """
    mean_ndci = img.select('NDCI').reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=20,             # match red-edge native resolution
        maxPixels=1e9,
        bestEffort=True
    ).get('NDCI')

    props = {
        'datetime': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd HH:mm:ss'),
        'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
        'time': ee.Date(img.get('system:time_start')).format('HH:mm:ss'),
        'ndci': mean_ndci,
        'sensor': tag
    }
    return ee.Feature(None, props)


In [ ]:
# -----------------------------------------------------------------------------
# Process each lake
# -----------------------------------------------------------------------------
for lake in lakes:
    # Define ROI: 500 m x 500 m box centered at given coordinate
    center = ee.Geometry.Point([lake['lon'], lake['lat']])
    roi = center.buffer(half_size_m).bounds()

    tag = f"{lake['name']}_S2"

    # Build collection --> preprocess --> add NDCI
    processed = (s2
        .filterDate(start_date, end_date)
        .filterBounds(roi)
        .map(preprocess_s2)
        .map(add_ndci))

    # Map images to features (per-scene ROI mean), then drop null NDCI values
    fc = ee.FeatureCollection(
        processed.map(lambda img: img_to_feature(img, roi, tag))
    ).filter(ee.Filter.notNull(['ndci']))

    # Print scene count
    count = fc.size().getInfo()
    print(lake['name'], 'valid Sentinel-2 scenes =', count)

    # Client-side CSV export
    rows = fc.getInfo()['features']
    records = [f['properties'] for f in rows]
    df = pd.DataFrame.from_records(records)
    df = df.sort_values('datetime') if 'datetime' in df.columns else df
    df.to_csv(lake['export_id'] + '.csv', index=False)

print("Done!")